Every cleaning decision must answer **three questions:**
1. **Is this value invalid or just rare?**
1. **Changing meaning or only fixing representation?**
1. **Could this introduce bias or leakage?**

### Handling Missing Values
#### Step 1: Classify missingness
**MCAR — Missing Completely At Random**
**Meaning:**
Data is missing for reasons **completely unrelated** to any variable in the dataset (observed or unobserved).

**Example:**
A sensor randomly malfunctions and fails to record temperature.

**Why it matters:**
- The missingness is pure noise.
- Removing missing rows or using simple imputation does **not bias** results.
- This is the *best-case scenario.*

***

**MAR — Missing At Random**
**Meaning:**
Data is missing **because of other observed variables,** but **not because of the missing value itself.**

**Example:**
Salary is missing more often for interns, and “intern” status is recorded.

**Why it matters:**
- Missingness is *systematic* but explainable.
- If you model or condition on the observed variables, you can still get unbiased estimates.
- Most modern imputation methods (regression, multiple imputation) assume MAR.

***

**MNAR — Missing Not At Random**
**Meaning:**
Data is missing **because of the value itself,** even after considering all other variables.

**Example:**
High earners choose not to report their salary.

**Why it matters:**
- The missingness contains information.
- Standard imputation or deletion will **bias results**.
- Requires specialized models or external assumptions.
- This is the hardest case to handle.

***

| Type | Meaning                 | Example                    |
| ---- | ----------------------- | -------------------------- |
| MCAR | Completely random       | Sensor glitch              |
| MAR  | Depends on other fields | Salary missing for interns |
| MNAR | Depends on itself       | High earners hide salary   |


#### Step 2: Decide per column
```python
df.isna().mean().sort_values(ascending=False)
```

#### Safe patterns
**Drop only when unavoidable**
```python
df = df.dropna(subset=["user_id"])
```

**Conditional imputation**
```python
df["salary"] = df.groupby("city")["salary"].transform(
    lambda x: x.fillna(x.median())
)
```

### Numeric Cleaning (Ranges, Outliers)
#### Validate before fixing
```python
assert (df["age"] >= 0).all()
```

#### Dangerous
```python
df["age"] = df["age"].clip(0, 120)
```

#### Safer pattern
```python
invalid_age = ~df["age"].between(0, 120)
df.loc[invalid_age, "age"] = np.nan
```


### Categorical Cleaning
#### Normalize representation
```python
df["city"] = (
    df["city"]
    .str.strip()
    .str.title()
)
```

#### Handle unknowns explicitly
```python
valid_cities = {"Pune", "Mumbai", "Delhi"}

df.loc[~df["city"].isin(valid_cities), "city"] = "Unknown"
```

`"Unknown"` must be **intentional**, not accidental.

### Datetime Cleaning
#### Always coerce
```python
df["signup_date"] = pd.to_datetime(
    df["signup_date"],
    errors="coerce"
)
```

#### Remove future leakage
```python
df = df[df["signup_date"] <= pd.Timestamp.today()]
```

Never “fix” future dates by shifting them.

### Deduplication
#### Identify duplicates
```python
df[df.duplicated(subset=["user_id"], keep=False)]
```

**Decide Strategy**
| Case               | Action      |
| ------------------ | ----------- |
| Exact duplicates   | Drop        |
| Conflicting rows   | Aggregate   |
| Partial duplicates | Manual rule |

```python
df = (
    df.sort_values("updated_at")
      .drop_duplicates("user_id", keep="last")
)
```


### Type Stabilization
```python
df["age"] = df["age"].astype("Int64")
df["city"] = df["city"].astype("category")
```

### Cleaning Pipelines
#### Bad
```python
d["age"].fillna(df["age"].median(), inplace=True)
```

#### Good
```python
def clean_age(df):
    invalid = ~df["age"].between(0, 120)
    df.loc[invalid, "age"] = np.nan
    df["age"] = df["age"].fillna(df["age"].median())
    return df
```


### Base Dataset

In [1]:
import pandas as pd
import numpy as np

df = pd.DataFrame({
    "user_id": [1, 2, 2, 4, 5, None],
    "age": [25, -3, 30, 200, None, 40],
    "salary": [50000, None, -1000, 70000, 65000, None],
    "city": [" pune ", "Mumbai", "DELHI", "Puna", None, "Mumbai"],
    "signup_date": [
        "2024-01-01",
        "2024-01-05",
        "invalid_date",
        "2026-03-01",
        None,
        "2024-01-10"
    ]
})

### Exercise 1
1. Inspect:
```python
df.info()
df.describe()
```
2. List **all columns** that contain:
   - invalid values
   - missing values
   - suspicious ranges

In [2]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6 entries, 0 to 5
Data columns (total 5 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   user_id      5 non-null      float64
 1   age          5 non-null      float64
 2   salary       4 non-null      float64
 3   city         5 non-null      object 
 4   signup_date  5 non-null      object 
dtypes: float64(3), object(2)
memory usage: 372.0+ bytes


In [3]:
df.describe()

,user_id,age,salary
count,5.000000,5.000000,4.000000
mean,2.800000,58.400000,46000.000000
std,1.643168,80.748375,32465.366161
min,1.000000,-3.000000,-1000.000000
25%,2.000000,25.000000,37250.000000
50%,2.000000,30.000000,57500.000000
75%,4.000000,40.000000,66250.000000
max,5.000000,200.000000,70000.000000


**Observations**
| Column      | Issues Found                                               |
| ----------- | ---------------------------------------------------------- |
| user_id     | Missing value, duplicates                                  |
| age         | Negative value, very large value (200), missing            |
| salary      | Negative value, missing                                    |
| city        | Whitespace, casing issues, invalid value (`Puna`), missing |
| signup_date | Invalid string, missing value, future date                 |


### Exercise 2
1. Identify rows with:
   - missing `user_id`
   - duplicate `user_id`
2. Decide:
   - Should duplicates be dropped or resolved?
3. Implement your decision.

In [6]:
df[df["user_id"].isna()]

,user_id,age,salary,city,signup_date
5,NaN,40.0,NaN,Mumbai,2024-01-10


In [7]:
df[df["user_id"].duplicated(keep=False)]

,user_id,age,salary,city,signup_date
1,2.0,-3.0,NaN,Mumbai,2024-01-05
2,2.0,30.0,-1000.0,DELHI,invalid_date


**Decision (Important)**
- Missing `user_id` -> DROP ROW (cannot identify entity)
- Duplicate `user_id` -> Resolve later using business logic

In [8]:
df = df.dropna(subset=["user_id"])

### Exercise 3
Rules:
- Valid age range: `0–120`
- Invalid values -> convert to `NaN`
- Missing values -> fill with **median age**

Steps:
1. Detect invalid ages
2. Convert invalid -> `NaN`
3. Impute missing ages safely

In [9]:
invalid_age = ~df["age"].between(0, 120)

In [10]:
df.loc[invalid_age, "age"] = np.nan

In [11]:
age_median = df["age"].median()
df["age"] = df["age"].fillna(age_median)

### Exercise 4
Rules:
- Salary must be `>= 0`
- Missing salary:
   - If **city known** -> fill with city median
   - If **city unknown** -> leave as `NaN`

Steps:
1. Convert invalid salaries -> `NaN`
2. Apply conditional imputation

In [12]:
df.loc[df["salary"] < 0, "salary"] = np.nan

In [13]:
df["salary"] = df.groupby("city")["salary"].transform(
    lambda x: x.fillna(x.median())
)

/Users/jatinrokde/CodeBase/SynapseWorks/MLForge/.venv/lib/python3.13/site-packages/numpy/lib/_nanfunctions_impl.py:1214: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)
/Users/jatinrokde/CodeBase/SynapseWorks/MLForge/.venv/lib/python3.13/site-packages/numpy/lib/_nanfunctions_impl.py:1214: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)


### Exercise 5
Rules:
- Strip whitespace
- Normalize case (Title Case)
- Valid cities: `{Pune, Mumbai, Delhi}`
- All others -> `"Unknown"`

Steps:
1. Normalize text
2. Enforce allowed domain

In [14]:
df["city"] = (
    df["city"]
    .str.strip()
    .str.title()
)

In [15]:
valid_cities = {"Pune", "Mumbai", "Delhi"}

df.loc[~df["city"].isin(valid_cities), "city"] = "Unknown"


### Exercise 6
Rules:
- Parse `signup_date` safely
- Invalid dates -> `NaT`
- Future dates -> **remove rows**
- Missing dates -> **drop rows**

Steps:
1. Convert to datetime
2. Remove future + missing dates

In [16]:
df["signup_date"] = pd.to_datetime(
    df["signup_date"],
    errors="coerce"
)

In [17]:
df = df.dropna(subset=["signup_date"])

In [18]:
df = df[df["signup_date"] <= pd.Timestamp.today()]

### Exercise 7
Rules:
- One row per `user_id`
- Keep the **latest signup_date**

Steps:
1. Sort appropriately
2. Deduplicate using business logic

In [19]:
df = (
    df.sort_values("signup_date")
      .drop_duplicates("user_id", keep="last")
)

### Exercise 8
Assert:
1. `user_id` is unique & non-null
2. `age` is within valid range
3. `salary` is non-negative or NaN
4. `city` has no unexpected values
5. `signup_date` is valid & not future

In [20]:
assert df["user_id"].notna().all()
assert df["user_id"].is_unique
assert df["age"].between(0, 120).all()
assert (df["salary"].ge(0) | df["salary"].isna()).all()
assert set(df["city"]).issubset({"Pune", "Mumbai", "Delhi", "Unknown"})
assert pd.api.types.is_datetime64_any_dtype(df["signup_date"])
assert df["signup_date"].max() <= pd.Timestamp.today()